# Cross-Dataset Comparison: All Three Experiments

**Experiments:**
1. Kaggle Brain Tumour MRI Dataset (Cleaned Version 2)
2. Figshare Brain Tumor Dataset (Random Image-Level Split)
3. Figshare Brain Tumor Dataset (Patient-Level Split)

**Models:** ResNet50 and EfficientNetB0

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print('Imports loaded.')

## 1. Load Results

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

with open('/content/drive/MyDrive/kaggle_results.json', 'r') as f:
    kaggle = json.load(f)

with open('/content/drive/MyDrive/figshare_results.json', 'r') as f:
    figshare_random = json.load(f)

with open('/content/drive/MyDrive/figshare_patient_split_results.json', 'r') as f:
    figshare_patient = json.load(f)

print('All 3 files loaded from Google Drive!')

In [ ]:
print('=' * 70)
print('ALL THREE EXPERIMENTS LOADED SUCCESSFULLY')
print('=' * 70)
print(f'\n1. Kaggle (Cleaned V2):        {kaggle["total_train"]} train, {kaggle["total_test"]} test, {kaggle["num_classes"]} classes')
print(f'2. Figshare (Random Split):    {figshare_random["total_train"]} train, {figshare_random["total_test"]} test, {figshare_random["num_classes"]} classes')
print(f'3. Figshare (Patient Split):   {figshare_patient["total_train"]} train, {figshare_patient["total_test"]} test, {figshare_patient["num_classes"]} classes')

## 2. Overall Model Comparison Tables

In [ ]:
print('=' * 90)
print('RESNET50: CROSS-EXPERIMENT COMPARISON')
print('=' * 90)
print(f'{"Metric":<20} {"Kaggle (Cleaned)":<20} {"Figshare (Random)":<20} {"Figshare (Patient)":<20}')
print('-' * 90)

for metric in ['accuracy', 'precision', 'recall', 'f1']:
    k_val = kaggle['resnet'][metric]
    fr_val = figshare_random['resnet'][metric]
    fp_val = figshare_patient['resnet'][metric]
    print(f'{metric.capitalize():<20} {k_val:>17.2%}   {fr_val:>17.2%}   {fp_val:>17.2%}')

k_time = kaggle['resnet']['training_time_min']
fr_time = figshare_random['resnet']['training_time_min']
fp_time = figshare_patient['resnet']['training_time_min']
print(f'{"Training Time":<20} {k_time:>16.2f}m   {fr_time:>16.2f}m   {fp_time:>16.2f}m')

k_kappa = kaggle['resnet']['kappa']
fr_kappa = figshare_random['resnet']['kappa']
fp_kappa = figshare_patient['resnet']['kappa']
print(f'{"Cohen Kappa":<20} {k_kappa:>17.4f}   {fr_kappa:>17.4f}   {fp_kappa:>17.4f}')

In [ ]:
print('=' * 90)
print('EFFICIENTNETB0: CROSS-EXPERIMENT COMPARISON')
print('=' * 90)
print(f'{"Metric":<20} {"Kaggle (Cleaned)":<20} {"Figshare (Random)":<20} {"Figshare (Patient)":<20}')
print('-' * 90)

for metric in ['accuracy', 'precision', 'recall', 'f1']:
    k_val = kaggle['effnet'][metric]
    fr_val = figshare_random['effnet'][metric]
    fp_val = figshare_patient['effnet'][metric]
    print(f'{metric.capitalize():<20} {k_val:>17.2%}   {fr_val:>17.2%}   {fp_val:>17.2%}')

k_time = kaggle['effnet']['training_time_min']
fr_time = figshare_random['effnet']['training_time_min']
fp_time = figshare_patient['effnet']['training_time_min']
print(f'{"Training Time":<20} {k_time:>16.2f}m   {fr_time:>16.2f}m   {fp_time:>16.2f}m')

k_kappa = kaggle['effnet']['kappa']
fr_kappa = figshare_random['effnet']['kappa']
fp_kappa = figshare_patient['effnet']['kappa']
print(f'{"Cohen Kappa":<20} {k_kappa:>17.4f}   {fr_kappa:>17.4f}   {fp_kappa:>17.4f}')

## 3. Data Leakage Impact: Random vs Patient-Level Split

In [ ]:
print('=' * 70)
print('DATA LEAKAGE IMPACT: RANDOM SPLIT vs PATIENT-LEVEL SPLIT')
print('=' * 70)
print(f'\n{"Model":<18} {"Random Split":<15} {"Patient Split":<15} {"Difference":<15} {"Inflation":<12}')
print('-' * 70)

for model_name, model_key in [('ResNet50', 'resnet'), ('EfficientNetB0', 'effnet')]:
    random_acc = figshare_random[model_key]['accuracy']
    patient_acc = figshare_patient[model_key]['accuracy']
    diff = random_acc - patient_acc
    inflation = (diff / patient_acc) * 100
    print(f'{model_name:<18} {random_acc:>12.2%}   {patient_acc:>12.2%}   {diff:>+12.2%}   {inflation:>+10.1f}%')

print('\nConclusion: Random split inflates accuracy because the model memorises')
print('patient-specific features rather than learning tumour characteristics.')

## 4. Per-Class F1-Score: Random vs Patient Split (Figshare)

In [ ]:
figshare_classes = figshare_patient['class_names']

print('=' * 70)
print('PER-CLASS F1-SCORE: RANDOM vs PATIENT SPLIT (Figshare)')
print('=' * 70)
print(f'\n{"Class":<15} {"Model":<18} {"Random Split":<15} {"Patient Split":<15} {"Difference":<12}')
print('-' * 70)

for i, cls in enumerate(figshare_classes):
    for model_name, model_key in [('ResNet50', 'resnet'), ('EfficientNetB0', 'effnet')]:
        random_f1 = figshare_random[model_key]['per_class_f1'][i]
        patient_f1 = figshare_patient[model_key]['per_class_f1'][i]
        diff = random_f1 - patient_f1
        print(f'{cls:<15} {model_name:<18} {random_f1:>12.2%}   {patient_f1:>12.2%}   {diff:>+10.2%}')
    print()

## 5. Figure: Accuracy Comparison Across All Experiments

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

experiments = ['Kaggle\n(Cleaned V2)', 'Figshare\n(Random Split)', 'Figshare\n(Patient Split)']
resnet_accs = [kaggle['resnet']['accuracy'] * 100,
               figshare_random['resnet']['accuracy'] * 100,
               figshare_patient['resnet']['accuracy'] * 100]
effnet_accs = [kaggle['effnet']['accuracy'] * 100,
               figshare_random['effnet']['accuracy'] * 100,
               figshare_patient['effnet']['accuracy'] * 100]

x = np.arange(len(experiments))
width = 0.3

bars1 = ax.bar(x - width/2, resnet_accs, width, label='ResNet50', color='steelblue', alpha=0.85)
bars2 = ax.bar(x + width/2, effnet_accs, width, label='EfficientNetB0', color='coral', alpha=0.85)

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.3,
                f'{height:.2f}%', ha='center', va='bottom', fontweight='bold', fontsize=10)

ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Model Accuracy Across All Three Experiments', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(experiments, fontsize=11)
ax.legend(fontsize=11)
ax.set_ylim(75, 100)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('accuracy_comparison_all_experiments.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Figure: Data Leakage Impact (Random vs Patient)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

metrics = ['accuracy', 'precision', 'recall', 'f1']
metric_labels = ['Accuracy', 'Precision', 'Recall', 'F1-Score']

# ResNet50
random_vals = [figshare_random['resnet'][m] * 100 for m in metrics]
patient_vals = [figshare_patient['resnet'][m] * 100 for m in metrics]

x = np.arange(len(metrics))
width = 0.3

bars1 = axes[0].bar(x - width/2, random_vals, width, label='Random Split', color='#ff6b6b', alpha=0.85)
bars2 = axes[0].bar(x + width/2, patient_vals, width, label='Patient Split', color='#4ecdc4', alpha=0.85)

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        axes[0].text(bar.get_x() + bar.get_width()/2., height + 0.3,
                     f'{height:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

axes[0].set_title('ResNet50: Random vs Patient Split', fontsize=13, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metric_labels)
axes[0].set_ylim(75, 100)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# EfficientNetB0
random_vals = [figshare_random['effnet'][m] * 100 for m in metrics]
patient_vals = [figshare_patient['effnet'][m] * 100 for m in metrics]

bars1 = axes[1].bar(x - width/2, random_vals, width, label='Random Split', color='#ff6b6b', alpha=0.85)
bars2 = axes[1].bar(x + width/2, patient_vals, width, label='Patient Split', color='#4ecdc4', alpha=0.85)

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        axes[1].text(bar.get_x() + bar.get_width()/2., height + 0.3,
                     f'{height:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

axes[1].set_title('EfficientNetB0: Random vs Patient Split', fontsize=13, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(metric_labels)
axes[1].set_ylim(75, 100)
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Impact of Data Leakage on Reported Performance (Figshare Dataset)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('data_leakage_impact.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Figure: Per-Class F1 Comparison (Figshare)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for idx, (model_name, model_key) in enumerate([('ResNet50', 'resnet'), ('EfficientNetB0', 'effnet')]):
    random_f1 = [f * 100 for f in figshare_random[model_key]['per_class_f1']]
    patient_f1 = [f * 100 for f in figshare_patient[model_key]['per_class_f1']]

    x = np.arange(len(figshare_classes))
    width = 0.3

    bars1 = axes[idx].bar(x - width/2, random_f1, width, label='Random Split', color='#ff6b6b', alpha=0.85)
    bars2 = axes[idx].bar(x + width/2, patient_f1, width, label='Patient Split', color='#4ecdc4', alpha=0.85)

    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            axes[idx].text(bar.get_x() + bar.get_width()/2., height + 0.3,
                          f'{height:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

    axes[idx].set_title(f'{model_name}: Per-Class F1-Score', fontsize=13, fontweight='bold')
    axes[idx].set_xticks(x)
    axes[idx].set_xticklabels(figshare_classes)
    axes[idx].set_ylim(70, 100)
    axes[idx].set_ylabel('F1-Score (%)')
    axes[idx].legend()
    axes[idx].grid(axis='y', alpha=0.3)

plt.suptitle('Per-Class Performance: Random vs Patient-Level Split',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('per_class_f1_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Figure: Complete Performance Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

row_labels = []
data = []

for model_name, model_key in [('ResNet50', 'resnet'), ('EfficientNetB0', 'effnet')]:
    for exp_name, exp_data in [('Kaggle', kaggle),
                                ('Figshare Random', figshare_random),
                                ('Figshare Patient', figshare_patient)]:
        row_labels.append(f'{model_name}\n{exp_name}')
        data.append([
            exp_data[model_key]['accuracy'] * 100,
            exp_data[model_key]['precision'] * 100,
            exp_data[model_key]['recall'] * 100,
            exp_data[model_key]['f1'] * 100,
            exp_data[model_key]['kappa'] * 100,
        ])

data = np.array(data)
col_labels = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Kappa\u00d7100']

sns.heatmap(data, annot=True, fmt='.2f', cmap='YlOrRd',
            xticklabels=col_labels, yticklabels=row_labels,
            vmin=75, vmax=100, ax=ax, linewidths=0.5)

ax.set_title('Complete Performance Comparison Across All Experiments',
             fontsize=14, fontweight='bold', pad=15)

plt.tight_layout()
plt.savefig('full_comparison_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

## 9. Summary of Key Findings

In [ ]:
print('=' * 70)
print('SUMMARY OF KEY FINDINGS')
print('=' * 70)

for model_name, model_key in [('ResNet50', 'resnet'), ('EfficientNetB0', 'effnet')]:
    random_acc = figshare_random[model_key]['accuracy'] * 100
    patient_acc = figshare_patient[model_key]['accuracy'] * 100
    diff = random_acc - patient_acc
    print(f'\n{model_name}:')
    print(f'  Kaggle (Cleaned V2):     {kaggle[model_key]["accuracy"]*100:.2f}%')
    print(f'  Figshare Random Split:   {random_acc:.2f}%')
    print(f'  Figshare Patient Split:  {patient_acc:.2f}%')
    print(f'  Leakage Inflation:       {diff:+.2f}% accuracy')

print(f'\nKey Contribution:')
print(f'  Random splitting inflates reported accuracy by ~{abs(figshare_random["resnet"]["accuracy"] - figshare_patient["resnet"]["accuracy"])*100:.1f}-{abs(figshare_random["effnet"]["accuracy"] - figshare_patient["effnet"]["accuracy"])*100:.1f}%')
print(f'  due to data leakage from same-patient slices in train and test sets.')
print(f'  Prior studies using random splits on Figshare report inflated results.')

print('\nFigures saved:')
print('  1. accuracy_comparison_all_experiments.png')
print('  2. data_leakage_impact.png')
print('  3. per_class_f1_comparison.png')
print('  4. full_comparison_heatmap.png')

## 10. System Configuration Table

In [ ]:
## System Configuration

print('=' * 60)
print('SYSTEM CONFIGURATION')
print('=' * 60)

config = [
    ('Platform', 'Google Colab'),
    ('GPU', 'NVIDIA Tesla T4 (15GB VRAM)'),
    ('System RAM', '~25GB'),
    ('Python', '3.10'),
    ('PyTorch', '2.5.1'),
    ('timm', '1.0+'),
    ('SHAP', 'Latest'),
    ('pytorch-grad-cam', 'Latest'),
    ('scikit-learn', 'Latest'),
    ('Random Seed', '42'),
]

print(f'\n{"Parameter":<25} {"Value":<35}')
print('-' * 60)
for param, val in config:
    print(f'{param:<25} {val:<35}')
print('=' * 60)

## 11. Hyperparameters Table

In [ ]:
## Training Hyperparameters

print('=' * 70)
print('TRAINING HYPERPARAMETERS')
print('=' * 70)

hyperparams = [
    ('Input Size', '224 × 224 × 3', '224 × 224 × 3'),
    ('Batch Size', '32', '32'),
    ('Max Epochs', '30', '30'),
    ('Early Stopping', '5 epochs patience', '5 epochs patience'),
    ('Learning Rate', '1e-4', '5e-5'),
    ('Optimiser', 'Adam', 'Adam'),
    ('LR Scheduler', 'ReduceLROnPlateau', 'ReduceLROnPlateau'),
    ('Scheduler Patience', '5 epochs', '5 epochs'),
    ('Scheduler Factor', '0.5', '0.5'),
    ('Min LR', '1e-7', '1e-7'),
    ('Dropout', '0.5', '0.5'),
    ('Loss Function', 'Weighted CrossEntropy', 'Weighted CrossEntropy'),
    ('Fine-tuning', 'Full (all layers)', 'Full (all layers)'),
    ('Normalisation', 'ImageNet stats', 'ImageNet stats'),
]

print(f'\n{"Hyperparameter":<25} {"ResNet50":<25} {"EfficientNetB0":<25}')
print('-' * 70)
for param, resnet, effnet in hyperparams:
    print(f'{param:<25} {resnet:<25} {effnet:<25}')
print('=' * 70)

## 12. Model Architecture Details

In [ ]:
## Model Architecture Details

print('=' * 75)
print('MODEL ARCHITECTURE DETAILS')
print('=' * 75)

arch = [
    ('Base Architecture', 'ResNet50', 'EfficientNetB0'),
    ('Pre-trained Weights', 'ImageNet', 'ImageNet'),
    ('Feature Dim', '2,048', '1,280'),
    ('FC Layer', '2048 → 256', '1280 → 256'),
    ('Activation', 'ReLU', 'ReLU'),
    ('Dropout', '0.5', '0.5'),
    ('Output Layer (Kaggle)', '256 → 4', '256 → 4'),
    ('Output Layer (Figshare)', '256 → 3', '256 → 3'),
    ('Total Params', '23,979,012', '4,377,220'),
    ('Trainable Params', '23,979,012 (100%)', '4,377,220 (100%)'),
    ('Param Reduction', '—', '82% fewer vs ResNet50'),
]

print(f'\n{"Property":<25} {"ResNet50":<25} {"EfficientNetB0":<25}')
print('-' * 75)
for prop, resnet, effnet in arch:
    print(f'{prop:<25} {resnet:<25} {effnet:<25}')
print('=' * 75)

## 13. Cross-Dataset Class Distribution

In [ ]:
## Cross-Dataset Class Distribution

print('=' * 85)
print('DISTRIBUTION OF DATASET SAMPLES ACROSS BRAIN TUMOUR CLASSES')
print('=' * 85)

print(f'\n{"Dataset":<25} {"Class":<15} {"Train":<10} {"Test":<10} {"Total":<10} {"% of Data":<10}')
print('-' * 85)

# Kaggle
kaggle_classes = kaggle['class_names']
for i, cls in enumerate(kaggle_classes):
    k_train_total = kaggle['total_train']
    k_test_total = kaggle['total_test']
    # Kaggle is balanced: estimate per-class
    k_train_per = k_train_total // len(kaggle_classes)
    k_test_per = k_test_total // len(kaggle_classes)
    k_total = k_train_per + k_test_per
    k_grand = k_train_total + k_test_total
    pct = k_total / k_grand * 100
    print(f'{"Kaggle (Cleaned V2)":<25} {cls:<15} {k_train_per:<10} {k_test_per:<10} {k_total:<10} {pct:<9.1f}%')
print(f'{"":25} {"TOTAL":<15} {kaggle["total_train"]:<10} {kaggle["total_test"]:<10} {kaggle["total_train"]+kaggle["total_test"]:<10}')

print()

# Figshare Random
fig_r_classes = figshare_random['class_names']
fig_r_grand = figshare_random['total_train'] + figshare_random['total_test']
for i, cls in enumerate(fig_r_classes):
    r_prec = figshare_random['resnet']['per_class_precision'][i]
    print(f'{"Figshare (Random)":<25} {cls:<15} {"—":<10} {"—":<10} {"—":<10} {"—":<10}')
print(f'{"":25} {"TOTAL":<15} {figshare_random["total_train"]:<10} {figshare_random["total_test"]:<10} {fig_r_grand:<10}')

print()

# Figshare Patient
print(f'{"Figshare (Patient)":<25} {"Glioma":<15} {1178:<10} {248:<10} {1426:<10} {1426/3064*100:<9.1f}%')
print(f'{"":25} {"Meningioma":<15} {565:<10} {143:<10} {708:<10} {708/3064*100:<9.1f}%')
print(f'{"":25} {"Pituitary":<15} {729:<10} {201:<10} {930:<10} {930/3064*100:<9.1f}%')
print(f'{"":25} {"TOTAL":<15} {2472:<10} {592:<10} {3064:<10}')

print('\n' + '=' * 85)

## 14. Performance Analysis Both Datasets Table

In [ ]:
## Performance Analysis: Both Models on All Datasets

print('=' * 95)
print('PERFORMANCE ANALYSIS: BOTH MODELS ACROSS ALL EXPERIMENTS')
print('=' * 95)

print(f'\n{"Experiment":<25} {"Model":<18} {"Acc":<10} {"Prec":<10} {"Recall":<10} {"F1":<10} {"Kappa":<10}')
print('-' * 95)

experiments = [
    ('Kaggle (Cleaned V2)', kaggle),
    ('Figshare (Random)', figshare_random),
    ('Figshare (Patient)', figshare_patient),
]

for exp_name, exp_data in experiments:
    for model_name, model_key in [('ResNet50', 'resnet'), ('EfficientNetB0', 'effnet')]:
        m = exp_data[model_key]
        print(f'{exp_name:<25} {model_name:<18} {m["accuracy"]:>7.2%}   {m["precision"]:>7.2%}   {m["recall"]:>7.2%}   {m["f1"]:>7.2%}   {m["kappa"]:>7.4f}')
    print()

print('=' * 95)

# Best model per experiment
print('\nBest Model per Experiment:')
for exp_name, exp_data in experiments:
    r_acc = exp_data['resnet']['accuracy']
    e_acc = exp_data['effnet']['accuracy']
    best = 'ResNet50' if r_acc > e_acc else 'EfficientNetB0'
    diff = abs(r_acc - e_acc) * 100
    print(f'  {exp_name}: {best} (+{diff:.2f}%)')

## 15. Performance Comparison with Existing Studies

In [ ]:
## Performance Comparison with Existing Studies

print('=' * 95)
print('PERFORMANCE COMPARISON WITH EXISTING STUDIES')
print('=' * 95)

existing_studies = [
    ('Musthafa et al. (2024)', 'ResNet50 + Grad-CAM', 'Figshare', 'Random', '97.33%'),
    ('Gencer & Gencer (2025)', 'EfficientNetB0 + QGA', 'Figshare', 'Random', '99.35%'),
    ('Ishaq et al. (2025)', 'EfficientNet (improved)', 'Kaggle V1', 'Random', '99.69%'),
    ('Afnaan et al. (2025)', 'ResNet50 + Grad-CAM', 'Kaggle V1', 'Pre-defined', '98.09%'),
    ('Bilal & Asif (2025)', 'VGG16 + Transfer', 'Figshare', 'Random', '98.32%'),
    ('Singh et al. (2025)', 'Fine-tuned CNNs', 'Multiple', 'Random', '95.74%'),
]

print(f'\n{"Study":<28} {"Method":<28} {"Dataset":<15} {"Split":<12} {"Accuracy":<10}')
print('-' * 95)

for study, method, dataset, split, acc in existing_studies:
    print(f'{study:<28} {method:<28} {dataset:<15} {split:<12} {acc:<10}')

print('-' * 95)

# Our results
our_results = [
    ('This Study', 'ResNet50', 'Kaggle V2 Clean', 'Pre-defined', f'{kaggle["resnet"]["accuracy"]*100:.2f}%'),
    ('This Study', 'EfficientNetB0', 'Kaggle V2 Clean', 'Pre-defined', f'{kaggle["effnet"]["accuracy"]*100:.2f}%'),
    ('This Study', 'ResNet50', 'Figshare', 'Random', f'{figshare_random["resnet"]["accuracy"]*100:.2f}%'),
    ('This Study', 'EfficientNetB0', 'Figshare', 'Random', f'{figshare_random["effnet"]["accuracy"]*100:.2f}%'),
    ('This Study', 'ResNet50', 'Figshare', 'Patient-level', f'{figshare_patient["resnet"]["accuracy"]*100:.2f}%'),
    ('This Study', 'EfficientNetB0', 'Figshare', 'Patient-level', f'{figshare_patient["effnet"]["accuracy"]*100:.2f}%'),
]

for study, method, dataset, split, acc in our_results:
    print(f'{study:<28} {method:<28} {dataset:<15} {split:<12} {acc:<10}')

print('=' * 95)

print('\nKey Observation:')
print('  Studies reporting 95%+ on Figshare used random image-level splits,')
print('  which causes data leakage. Our patient-level split produces lower')
print('  but more realistic accuracy that reflects true generalisation ability.')
print('  The difference demonstrates that prior results are inflated.')

In [ ]:
## Comprehensive Visual Comparison: All Three Experiments

fig, axes = plt.subplots(2, 3, figsize=(20, 12))

experiments = [
    ('Kaggle\n(Cleaned V2)', kaggle),
    ('Figshare\n(Random Split)', figshare_random),
    ('Figshare\n(Patient Split)', figshare_patient),
]

colors_resnet = '#2E86C1'
colors_effnet = '#E74C3C'

# ---- ROW 1: Per-Class F1-Score for each experiment ----
for col, (exp_name, exp_data) in enumerate(experiments):
    classes = exp_data['class_names']
    resnet_f1 = [f * 100 for f in exp_data['resnet']['per_class_f1']]
    effnet_f1 = [f * 100 for f in exp_data['effnet']['per_class_f1']]

    x = np.arange(len(classes))
    width = 0.35

    bars1 = axes[0][col].bar(x - width/2, resnet_f1, width, label='ResNet50', color=colors_resnet, alpha=0.85)
    bars2 = axes[0][col].bar(x + width/2, effnet_f1, width, label='EfficientNetB0', color=colors_effnet, alpha=0.85)

    for bars in [bars1, bars2]:
        for bar in bars:
            h = bar.get_height()
            axes[0][col].text(bar.get_x() + bar.get_width()/2., h + 0.5,
                              f'{h:.1f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

    axes[0][col].set_title(f'{exp_name}', fontsize=13, fontweight='bold')
    axes[0][col].set_xticks(x)
    axes[0][col].set_xticklabels(classes, fontsize=9)
    axes[0][col].set_ylim(60, 105)
    axes[0][col].set_ylabel('F1-Score (%)' if col == 0 else '')
    axes[0][col].legend(fontsize=8, loc='lower right')
    axes[0][col].grid(axis='y', alpha=0.3)

# ---- ROW 2: Overall Metrics for each experiment ----
metrics = ['accuracy', 'precision', 'recall', 'f1']
metric_labels = ['Acc', 'Prec', 'Recall', 'F1']

for col, (exp_name, exp_data) in enumerate(experiments):
    resnet_vals = [exp_data['resnet'][m] * 100 for m in metrics]
    effnet_vals = [exp_data['effnet'][m] * 100 for m in metrics]

    x = np.arange(len(metrics))
    width = 0.35

    bars1 = axes[1][col].bar(x - width/2, resnet_vals, width, label='ResNet50', color=colors_resnet, alpha=0.85)
    bars2 = axes[1][col].bar(x + width/2, effnet_vals, width, label='EfficientNetB0', color=colors_effnet, alpha=0.85)

    for bars in [bars1, bars2]:
        for bar in bars:
            h = bar.get_height()
            axes[1][col].text(bar.get_x() + bar.get_width()/2., h + 0.5,
                              f'{h:.1f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

    axes[1][col].set_title(f'{exp_name}', fontsize=13, fontweight='bold')
    axes[1][col].set_xticks(x)
    axes[1][col].set_xticklabels(metric_labels, fontsize=10)
    axes[1][col].set_ylim(75, 105)
    axes[1][col].set_ylabel('Score (%)' if col == 0 else '')
    axes[1][col].legend(fontsize=8, loc='lower right')
    axes[1][col].grid(axis='y', alpha=0.3)

fig.text(0.01, 0.75, 'Per-Class F1', fontsize=14, fontweight='bold', rotation=90, va='center')
fig.text(0.01, 0.28, 'Overall Metrics', fontsize=14, fontweight='bold', rotation=90, va='center')

plt.suptitle('Complete Performance Comparison Across All Three Experiments',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout(rect=[0.02, 0, 1, 1])
plt.savefig('all_experiments_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

metrics = ['accuracy', 'precision', 'recall', 'f1']
metric_labels = ['Accuracy', 'Precision', 'Recall', 'F1-Score']

x = np.arange(len(metrics))
width = 0.25

# ResNet50
kaggle_vals = [kaggle['resnet'][m] * 100 for m in metrics]
random_vals = [figshare_random['resnet'][m] * 100 for m in metrics]
patient_vals = [figshare_patient['resnet'][m] * 100 for m in metrics]

bars1 = axes[0].bar(x - width, kaggle_vals, width, label='Kaggle (Cleaned V2)', color='#3498db', alpha=0.85)
bars2 = axes[0].bar(x, random_vals, width, label='Figshare (Random Split)', color='#ff6b6b', alpha=0.85)
bars3 = axes[0].bar(x + width, patient_vals, width, label='Figshare (Patient Split)', color='#4ecdc4', alpha=0.85)

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        axes[0].text(bar.get_x() + bar.get_width()/2., height + 0.3,
                     f'{height:.1f}%', ha='center', va='bottom', fontsize=8, fontweight='bold')

axes[0].set_title('ResNet50', fontsize=13, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metric_labels)
axes[0].set_ylim(75, 105)
axes[0].legend(fontsize=9, loc='lower right')
axes[0].grid(axis='y', alpha=0.3)
axes[0].set_ylabel('Score (%)', fontsize=11)

# EfficientNetB0
kaggle_vals = [kaggle['effnet'][m] * 100 for m in metrics]
random_vals = [figshare_random['effnet'][m] * 100 for m in metrics]
patient_vals = [figshare_patient['effnet'][m] * 100 for m in metrics]

bars1 = axes[1].bar(x - width, kaggle_vals, width, label='Kaggle (Cleaned V2)', color='#3498db', alpha=0.85)
bars2 = axes[1].bar(x, random_vals, width, label='Figshare (Random Split)', color='#ff6b6b', alpha=0.85)
bars3 = axes[1].bar(x + width, patient_vals, width, label='Figshare (Patient Split)', color='#4ecdc4', alpha=0.85)

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        axes[1].text(bar.get_x() + bar.get_width()/2., height + 0.3,
                     f'{height:.1f}%', ha='center', va='bottom', fontsize=8, fontweight='bold')

axes[1].set_title('EfficientNetB0', fontsize=13, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(metric_labels)
axes[1].set_ylim(75, 105)
axes[1].legend(fontsize=9, loc='lower right')
axes[1].grid(axis='y', alpha=0.3)
axes[1].set_ylabel('Score (%)', fontsize=11)

plt.suptitle('Performance Comparison Across All Three Experiments',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('all_experiments_performance.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Shared classes between Kaggle and Figshare
shared_classes = ['Glioma', 'Meningioma', 'Pituitary']

# Map class index for each dataset
kaggle_class_idx = {cls.lower(): i for i, cls in enumerate(kaggle['class_names'])}
figshare_class_idx = {cls: i for i, cls in enumerate(figshare_patient['class_names'])}

for idx, (model_name, model_key) in enumerate([('ResNet50', 'resnet'), ('EfficientNetB0', 'effnet')]):
    kaggle_f1 = [kaggle[model_key]['per_class_f1'][kaggle_class_idx[cls.lower()]] * 100 for cls in shared_classes]
    random_f1 = [figshare_random[model_key]['per_class_f1'][figshare_class_idx[cls]] * 100 for cls in shared_classes]
    patient_f1 = [figshare_patient[model_key]['per_class_f1'][figshare_class_idx[cls]] * 100 for cls in shared_classes]

    x = np.arange(len(shared_classes))
    width = 0.25

    bars1 = axes[idx].bar(x - width, kaggle_f1, width, label='Kaggle (Cleaned V2)', color='#3498db', alpha=0.85)
    bars2 = axes[idx].bar(x, random_f1, width, label='Figshare (Random)', color='#ff6b6b', alpha=0.85)
    bars3 = axes[idx].bar(x + width, patient_f1, width, label='Figshare (Patient)', color='#4ecdc4', alpha=0.85)

    for bars in [bars1, bars2, bars3]:
        for bar in bars:
            height = bar.get_height()
            axes[idx].text(bar.get_x() + bar.get_width()/2., height + 0.3,
                          f'{height:.1f}%', ha='center', va='bottom', fontsize=8, fontweight='bold')

    axes[idx].set_title(f'{model_name}: Per-Class F1-Score', fontsize=13, fontweight='bold')
    axes[idx].set_xticks(x)
    axes[idx].set_xticklabels(shared_classes, fontsize=11)
    axes[idx].set_ylim(60, 105)
    axes[idx].set_ylabel('F1-Score (%)' if idx == 0 else '')
    axes[idx].legend(fontsize=9, loc='lower right')
    axes[idx].grid(axis='y', alpha=0.3)

plt.suptitle('Per-Class F1-Score Across All Three Experiments\n(3 shared classes: Glioma, Meningioma, Pituitary — Kaggle also has No Tumour)',
             fontsize=13, fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig('per_class_f1_all_experiments.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
print("Kaggle history_resnet keys:", list(kaggle['history_resnet'].keys()))
print("Kaggle history_effnet keys:", list(kaggle['history_effnet'].keys()))

print("\nFirst 3 values of each:")
for key, vals in kaggle['history_resnet'].items():
    print(f"  {key}: {vals[:3]}")

In [ ]:
## Training Curves: All 6 Experiments (2 Models × 3 Datasets)

fig, axes = plt.subplots(2, 3, figsize=(20, 12))

experiments = [
    ('Kaggle (Cleaned V2)', kaggle),
    ('Figshare (Random Split)', figshare_random),
    ('Figshare (Patient Split)', figshare_patient),
]

colors = {
    'resnet_train': '#2E86C1',
    'resnet_test': '#85C1E9',
    'effnet_train': '#E74C3C',
    'effnet_test': '#F1948A',
}

for col, (exp_name, exp_data) in enumerate(experiments):
    resnet_epochs = range(1, len(exp_data['history_resnet']['train_acc']) + 1)
    effnet_epochs = range(1, len(exp_data['history_effnet']['train_acc']) + 1)

    # ---- ROW 1: Accuracy ----
    axes[0][col].plot(resnet_epochs, exp_data['history_resnet']['train_acc'],
                      color=colors['resnet_train'], linewidth=2, label='ResNet50 Train')
    axes[0][col].plot(resnet_epochs, exp_data['history_resnet']['test_acc'],
                      color=colors['resnet_test'], linewidth=2, linestyle='--', label='ResNet50 Test')
    axes[0][col].plot(effnet_epochs, exp_data['history_effnet']['train_acc'],
                      color=colors['effnet_train'], linewidth=2, label='EffNetB0 Train')
    axes[0][col].plot(effnet_epochs, exp_data['history_effnet']['test_acc'],
                      color=colors['effnet_test'], linewidth=2, linestyle='--', label='EffNetB0 Test')

    # Mark best test accuracy
    best_resnet_acc = max(exp_data['history_resnet']['test_acc'])
    best_resnet_epoch = exp_data['history_resnet']['test_acc'].index(best_resnet_acc) + 1
    best_effnet_acc = max(exp_data['history_effnet']['test_acc'])
    best_effnet_epoch = exp_data['history_effnet']['test_acc'].index(best_effnet_acc) + 1

    axes[0][col].scatter(best_resnet_epoch, best_resnet_acc, color=colors['resnet_train'],
                         s=100, zorder=5, marker='*')
    axes[0][col].scatter(best_effnet_epoch, best_effnet_acc, color=colors['effnet_train'],
                         s=100, zorder=5, marker='*')

    axes[0][col].set_title(f'{exp_name}', fontsize=13, fontweight='bold')
    axes[0][col].set_xlabel('Epoch')
    axes[0][col].set_ylabel('Accuracy (%)' if col == 0 else '')
    axes[0][col].legend(fontsize=8, loc='lower right')
    axes[0][col].grid(alpha=0.3)
    axes[0][col].set_ylim(40, 102)

    axes[0][col].text(0.02, 0.02,
                      f'Best: R={best_resnet_acc:.2f}% (ep{best_resnet_epoch})\n'
                      f'Best: E={best_effnet_acc:.2f}% (ep{best_effnet_epoch})',
                      transform=axes[0][col].transAxes, fontsize=8,
                      verticalalignment='bottom', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    # ---- ROW 2: Loss ----
    axes[1][col].plot(resnet_epochs, exp_data['history_resnet']['train_loss'],
                      color=colors['resnet_train'], linewidth=2, label='ResNet50 Train')
    axes[1][col].plot(resnet_epochs, exp_data['history_resnet']['test_loss'],
                      color=colors['resnet_test'], linewidth=2, linestyle='--', label='ResNet50 Test')
    axes[1][col].plot(effnet_epochs, exp_data['history_effnet']['train_loss'],
                      color=colors['effnet_train'], linewidth=2, label='EffNetB0 Train')
    axes[1][col].plot(effnet_epochs, exp_data['history_effnet']['test_loss'],
                      color=colors['effnet_test'], linewidth=2, linestyle='--', label='EffNetB0 Test')

    best_resnet_loss = min(exp_data['history_resnet']['test_loss'])
    best_resnet_loss_epoch = exp_data['history_resnet']['test_loss'].index(best_resnet_loss) + 1
    best_effnet_loss = min(exp_data['history_effnet']['test_loss'])
    best_effnet_loss_epoch = exp_data['history_effnet']['test_loss'].index(best_effnet_loss) + 1

    axes[1][col].scatter(best_resnet_loss_epoch, best_resnet_loss, color=colors['resnet_train'],
                         s=100, zorder=5, marker='*')
    axes[1][col].scatter(best_effnet_loss_epoch, best_effnet_loss, color=colors['effnet_train'],
                         s=100, zorder=5, marker='*')

    axes[1][col].set_title(f'{exp_name}', fontsize=13, fontweight='bold')
    axes[1][col].set_xlabel('Epoch')
    axes[1][col].set_ylabel('Loss' if col == 0 else '')
    axes[1][col].legend(fontsize=8, loc='upper right')
    axes[1][col].grid(alpha=0.3)

    axes[1][col].text(0.02, 0.98,
                      f'Best: R={best_resnet_loss:.4f} (ep{best_resnet_loss_epoch})\n'
                      f'Best: E={best_effnet_loss:.4f} (ep{best_effnet_loss_epoch})',
                      transform=axes[1][col].transAxes, fontsize=8,
                      verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.suptitle('Training Curves: All 6 Experiments (2 Models × 3 Datasets)\n★ = Best epoch',
             fontsize=15, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig('training_curves_all_experiments.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
print("Kaggle resnet keys:", list(kaggle['resnet'].keys()))

In [ ]:
## Cohen's Kappa Analysis: Figshare Patient-Level Split

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

figshare_classes = figshare_patient['class_names']

# ---- Chart 1: Overall Kappa Comparison ----
models = ['ResNet50', 'EfficientNetB0']
kappas = [figshare_patient['resnet']['kappa'], figshare_patient['effnet']['kappa']]
bar_colors = ['#2E86C1', '#E74C3C']

bars = axes[0].bar(models, kappas, color=bar_colors, alpha=0.85, width=0.5)
for bar in bars:
    h = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width() / 2., h + 0.005,
                 f'{h:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

axes[0].set_title("Overall Cohen's Kappa", fontsize=13, fontweight='bold')
axes[0].set_ylabel('Kappa Score')
axes[0].set_ylim(0, 1.1)
axes[0].grid(axis='y', alpha=0.3)
axes[0].axhline(y=0.81, color='green', linestyle='--', alpha=0.5, label='Almost Perfect (0.81)')
axes[0].axhline(y=0.61, color='orange', linestyle='--', alpha=0.5, label='Substantial (0.61)')
axes[0].legend(fontsize=8, loc='lower right')

# ---- Chart 2: Per-Class Kappa ----
x = np.arange(len(figshare_classes))
width = 0.35

resnet_kappa = figshare_patient['resnet']['per_class_kappa']
effnet_kappa = figshare_patient['effnet']['per_class_kappa']

bars1 = axes[1].bar(x - width / 2, resnet_kappa, width, label='ResNet50', color='#2E86C1', alpha=0.85)
bars2 = axes[1].bar(x + width / 2, effnet_kappa, width, label='EfficientNetB0', color='#E74C3C', alpha=0.85)

for bars in [bars1, bars2]:
    for bar in bars:
        h = bar.get_height()
        axes[1].text(bar.get_x() + bar.get_width() / 2., h + 0.005,
                     f'{h:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

axes[1].set_title("Per-Class Cohen's Kappa", fontsize=13, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(figshare_classes)
axes[1].set_ylabel('Kappa Score')
axes[1].set_ylim(0, 1.1)
axes[1].legend(fontsize=9)
axes[1].grid(axis='y', alpha=0.3)
axes[1].axhline(y=0.81, color='green', linestyle='--', alpha=0.5)
axes[1].axhline(y=0.61, color='orange', linestyle='--', alpha=0.5)

# ---- Chart 3: Kappa Across All 3 Experiments ----
exp_labels = ['Kaggle\n(Cleaned)', 'Figshare\n(Random)', 'Figshare\n(Patient)']
resnet_all = [kaggle['resnet']['kappa'], figshare_random['resnet']['kappa'], figshare_patient['resnet']['kappa']]
effnet_all = [kaggle['effnet']['kappa'], figshare_random['effnet']['kappa'], figshare_patient['effnet']['kappa']]

x2 = np.arange(len(exp_labels))

bars1 = axes[2].bar(x2 - width / 2, resnet_all, width, label='ResNet50', color='#2E86C1', alpha=0.85)
bars2 = axes[2].bar(x2 + width / 2, effnet_all, width, label='EfficientNetB0', color='#E74C3C', alpha=0.85)

for bars in [bars1, bars2]:
    for bar in bars:
        h = bar.get_height()
        axes[2].text(bar.get_x() + bar.get_width() / 2., h + 0.005,
                     f'{h:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

axes[2].set_title('Kappa Across All Experiments', fontsize=13, fontweight='bold')
axes[2].set_xticks(x2)
axes[2].set_xticklabels(exp_labels)
axes[2].set_ylabel('Kappa Score')
axes[2].set_ylim(0, 1.1)
axes[2].legend(fontsize=9)
axes[2].grid(axis='y', alpha=0.3)
axes[2].axhline(y=0.81, color='green', linestyle='--', alpha=0.5)
axes[2].axhline(y=0.61, color='orange', linestyle='--', alpha=0.5)

plt.suptitle("Cohen's Kappa Analysis: Figshare Dataset (Patient-Level Split)",
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('cohens_kappa_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# Interpretation Table
print('\n' + '=' * 70)
print("COHEN'S KAPPA INTERPRETATION GUIDE")
print('=' * 70)
print(f'  {"Range":<20} {"Interpretation":<25}')
print(f'  {"-" * 45}')
print(f'  {"< 0.20":<20} {"Poor":<25}')
print(f'  {"0.21 - 0.40":<20} {"Fair":<25}')
print(f'  {"0.41 - 0.60":<20} {"Moderate":<25}')
print(f'  {"0.61 - 0.80":<20} {"Substantial":<25}')
print(f'  {"0.81 - 1.00":<20} {"Almost Perfect":<25}')

print('\n' + '=' * 70)
print('KAPPA RESULTS SUMMARY')
print('=' * 70)

for exp_name, exp_data in [('Kaggle (Cleaned)', kaggle), ('Figshare (Random)', figshare_random), ('Figshare (Patient)', figshare_patient)]:
    print(f'\n{exp_name}:')
    for model_name, model_key in [('ResNet50', 'resnet'), ('EfficientNetB0', 'effnet')]:
        k = exp_data[model_key]['kappa']
        if k >= 0.81:
            interp = 'Almost Perfect'
        elif k >= 0.61:
            interp = 'Substantial'
        elif k >= 0.41:
            interp = 'Moderate'
        else:
            interp = 'Fair/Poor'
        print(f'  {model_name:<18} Kappa = {k:.4f} ({interp})')

    if 'inter_model_kappa' in exp_data:
        ik = exp_data['inter_model_kappa']
        print(f'  {"Inter-Model":<18} Kappa = {ik:.4f} (agreement between models)')

print('\n' + '=' * 70)

In [ ]:
## Computational Cost Analysis and Bubble Chart

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# ---- Chart 1: Training Time Comparison Across All Experiments ----
exp_labels = ['Kaggle\n(Cleaned)', 'Figshare\n(Random)', 'Figshare\n(Patient)']
resnet_times = [kaggle['resnet']['training_time_min'],
                figshare_random['resnet']['training_time_min'],
                figshare_patient['resnet']['training_time_min']]
effnet_times = [kaggle['effnet']['training_time_min'],
                figshare_random['effnet']['training_time_min'],
                figshare_patient['effnet']['training_time_min']]

x = np.arange(len(exp_labels))
width = 0.35

bars1 = axes[0].bar(x - width / 2, resnet_times, width, label='ResNet50', color='#2E86C1', alpha=0.85)
bars2 = axes[0].bar(x + width / 2, effnet_times, width, label='EfficientNetB0', color='#E74C3C', alpha=0.85)

for bars in [bars1, bars2]:
    for bar in bars:
        h = bar.get_height()
        axes[0].text(bar.get_x() + bar.get_width() / 2., h + 0.2,
                     f'{h:.1f}m', ha='center', va='bottom', fontsize=9, fontweight='bold')

axes[0].set_title('Training Time', fontsize=13, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(exp_labels)
axes[0].set_ylabel('Time (minutes)')
axes[0].legend(fontsize=9)
axes[0].grid(axis='y', alpha=0.3)

# ---- Chart 2: Inference Time Comparison ----
resnet_inf = [kaggle['resnet']['inference_ms'],
              figshare_random['resnet']['inference_ms'],
              figshare_patient['resnet']['inference_ms']]
effnet_inf = [kaggle['effnet']['inference_ms'],
              figshare_random['effnet']['inference_ms'],
              figshare_patient['effnet']['inference_ms']]

bars1 = axes[1].bar(x - width / 2, resnet_inf, width, label='ResNet50', color='#2E86C1', alpha=0.85)
bars2 = axes[1].bar(x + width / 2, effnet_inf, width, label='EfficientNetB0', color='#E74C3C', alpha=0.85)

for bars in [bars1, bars2]:
    for bar in bars:
        h = bar.get_height()
        axes[1].text(bar.get_x() + bar.get_width() / 2., h + 0.05,
                     f'{h:.2f}ms', ha='center', va='bottom', fontsize=9, fontweight='bold')

axes[1].set_title('Inference Time Per Image', fontsize=13, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(exp_labels)
axes[1].set_ylabel('Time (ms)')
axes[1].legend(fontsize=9)
axes[1].grid(axis='y', alpha=0.3)

# ---- Chart 3: Bubble Chart - Params vs Accuracy vs Training Time ----
params = {'ResNet50': 23.98, 'EfficientNetB0': 4.38}

for exp_name, exp_data, marker in [('Kaggle', kaggle, 'o'),
                                     ('Figshare Random', figshare_random, 's'),
                                     ('Figshare Patient', figshare_patient, 'D')]:
    for model_name, model_key, color in [('ResNet50', 'resnet', '#2E86C1'),
                                          ('EfficientNetB0', 'effnet', '#E74C3C')]:
        acc = exp_data[model_key]['accuracy'] * 100
        time_min = exp_data[model_key]['training_time_min']
        p = params[model_name]

        # Bubble size based on training time
        bubble_size = time_min * 15

        axes[2].scatter(p, acc, s=bubble_size, c=color, marker=marker,
                        alpha=0.7, edgecolors='black', linewidth=0.5,
                        label=f'{model_name} - {exp_name}')

        axes[2].annotate(f'{acc:.1f}%', (p, acc), textcoords='offset points',
                         xytext=(10, 5), fontsize=8, fontweight='bold')

axes[2].set_title('Parameters vs Accuracy\n(bubble size = training time)', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Parameters (Millions)')
axes[2].set_ylabel('Accuracy (%)')
axes[2].set_xlim(0, 28)
axes[2].set_ylim(80, 100)
axes[2].grid(alpha=0.3)

# Custom legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#2E86C1', markersize=10, label='ResNet50'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#E74C3C', markersize=10, label='EfficientNetB0'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='gray', markersize=8, label='Kaggle'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='gray', markersize=8, label='Figshare Random'),
    Line2D([0], [0], marker='D', color='w', markerfacecolor='gray', markersize=8, label='Figshare Patient'),
]
axes[2].legend(handles=legend_elements, fontsize=8, loc='lower right')

plt.suptitle('Computational Cost Analysis Across All Experiments',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('computational_cost_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# Summary Table
print('\n' + '=' * 90)
print('COMPUTATIONAL COST SUMMARY')
print('=' * 90)
print(f'\n{"Experiment":<25} {"Model":<18} {"Params (M)":<12} {"Train (min)":<14} {"Inference (ms)":<16} {"Accuracy":<10}')
print('-' * 90)

for exp_name, exp_data in [('Kaggle (Cleaned)', kaggle),
                            ('Figshare (Random)', figshare_random),
                            ('Figshare (Patient)', figshare_patient)]:
    for model_name, model_key in [('ResNet50', 'resnet'), ('EfficientNetB0', 'effnet')]:
        p = params[model_name]
        t = exp_data[model_key]['training_time_min']
        inf = exp_data[model_key]['inference_ms']
        acc = exp_data[model_key]['accuracy'] * 100
        print(f'{exp_name:<25} {model_name:<18} {p:<12.2f} {t:<14.2f} {inf:<16.2f} {acc:<9.2f}%')
    print()

print('-' * 90)
print('\nKey Findings:')

# Calculate efficiency
for exp_name, exp_data in [('Figshare (Patient)', figshare_patient)]:
    r_acc = exp_data['resnet']['accuracy'] * 100
    e_acc = exp_data['effnet']['accuracy'] * 100
    acc_diff = abs(r_acc - e_acc)
    param_reduction = (1 - params['EfficientNetB0'] / params['ResNet50']) * 100
    r_time = exp_data['resnet']['training_time_min']
    e_time = exp_data['effnet']['training_time_min']
    time_reduction = (1 - e_time / r_time) * 100 if r_time > e_time else -(e_time / r_time - 1) * 100

    print(f'  EfficientNetB0 has {param_reduction:.0f}% fewer parameters than ResNet50')
    print(f'  Accuracy difference on Patient Split: {acc_diff:.2f}%')
    print(f'  Training time difference: {abs(time_reduction):.1f}% {"faster" if time_reduction > 0 else "slower"} (EfficientNetB0)')
    print(f'  EfficientNetB0 offers {param_reduction:.0f}% parameter reduction with only {acc_diff:.2f}% accuracy trade-off')

print('\n' + '=' * 90)